In [18]:
from pathlib import Path
import duckdb
import numpy as np

print("1. Fetching unique customer list from 'paySim dataset.csv'...")

# Fetch all unique origin customers
unique_cust_df = duckdb.query("""
    SELECT DISTINCT nameOrig AS customer_id 
    FROM read_csv_auto('paySim dataset.csv')
""").df()

n_cust = len(unique_cust_df)
np.random.seed(42)

# Generate synthetic profile columns
unique_cust_df['account_type'] = np.random.choice(['SAVINGS', 'CHECKING', 'BUSINESS'], size=n_cust)
unique_cust_df['customer_segment'] = np.random.choice(['RETAIL', 'VIP', 'CORPORATE'], size=n_cust)
unique_cust_df['kyc_status'] = np.random.choice(['VERIFIED', 'PENDING'], size=n_cust, p=[0.85, 0.15])


unique_cust_df['risk_rating_base'] = np.random.choice(['LOW', 'MEDIUM', 'HIGH'], size=n_cust, p=[0.60, 0.30, 0.10])

# Save as CSV to Desktop
output_path = Path.home() / "Desktop" / "customer_profiles.csv"
unique_cust_df.to_csv(output_path, index=False)

print(f"-> Success: 'customer_profiles.csv' created for {n_cust:,} customers!")

1. Fetching unique customer list from 'paySim dataset.csv'...
-> Success: 'customer_profiles.csv' created for 6,353,307 customers!


In [20]:
import pandas as pd
import duckdb
import numpy as np

# 1. ADIM: Hem Girişi Hem Çıkışı Olan Gerçek Mule Hesapları Süzme
sql_mule_query = """
WITH CustomerTransactions AS (
    -- 1. Gönderen (Outflow) olarak yapılan işlemler
    SELECT 
        nameOrig AS customer_id,
        0 AS inflow,
        amount AS outflow
    FROM read_csv_auto('paySim dataset.csv')
    WHERE type IN ('TRANSFER', 'CASH_OUT', 'PAYMENT')

    UNION ALL

    -- 2. Alıcı (Inflow) olarak yapılan işlemler
    SELECT 
        nameDest AS customer_id,
        amount AS inflow,
        0 AS outflow
    FROM read_csv_auto('paySim dataset.csv')
    WHERE type IN ('CASH_IN', 'TRANSFER')
)
SELECT 
    t.customer_id,
    cp.account_type,
    cp.customer_segment,
    cp.kyc_status,
    cp.risk_rating_base,
    SUM(t.inflow) AS total_inflow,
    SUM(t.outflow) AS total_outflow,
    COUNT(*) AS total_txn_count
FROM CustomerTransactions t
LEFT JOIN read_csv_auto('customer_profiles.csv') cp
       ON t.customer_id = cp.customer_id
WHERE cp.customer_segment = 'RETAIL'
GROUP BY t.customer_id, cp.account_type, cp.customer_segment, cp.kyc_status, cp.risk_rating_base
-- KRİTİK ŞART: Hem girişi hem çıkışı olacak ve yüksek hacimli olacak
HAVING SUM(t.inflow) > 0 
   AND SUM(t.outflow) > 0 
   AND SUM(t.inflow) >= 50000
ORDER BY total_outflow DESC
"""

print("[1/2] DuckDB ile Gerçek Money Mule Sorgusu Çalıştırılıyor...")
mule_df = duckdb.query(sql_mule_query).df()

# 2. ADIM: Pandas Katmanı (Boşaltma Oranı & Risk Skorlama)
print("[2/2] Pandas Katmanında Risk Hesaplaması Yapılıyor...")

# Drain Percentage (Giren paranın yüzde kaçı çıkmış?)
mule_df['drain_percentage'] = (mule_df['total_outflow'] / mule_df['total_inflow']) * 100

# Nihai Risk Seviyesi Atama
def calculate_risk(row):
    if row['total_inflow'] >= 200000 or row['kyc_status'] == 'PENDING':
        return 'CRITICAL'
    elif row['total_inflow'] >= 100000 or row['risk_rating_base'] == 'HIGH':
        return 'HIGH'
    else:
        return 'MEDIUM'

mule_df['final_risk_level'] = mule_df.apply(calculate_risk, axis=1)

# Filtreleme: Yüksek Riskliler
retail_high_risk_mules = mule_df[
    mule_df['final_risk_level'].isin(['HIGH', 'CRITICAL'])
].copy()

# Kaydetme
retail_high_risk_mules.to_csv('AML_Alerts_Money_Mule_Retail_HighRisk.csv', index=False)

print(f"-> Harika! Toplam {len(retail_high_risk_mules)} adet GERÇEK (Giriş+Çıkış yapmış) yüksek riskli alarm süzüldü.")

retail_high_risk_mules.head(10)

[1/2] DuckDB ile Gerçek Money Mule Sorgusu Çalıştırılıyor...
[2/2] Pandas Katmanında Risk Hesaplaması Yapılıyor...
-> Harika! Toplam 346 adet GERÇEK (Giriş+Çıkış yapmış) yüksek riskli alarm süzüldü.


,customer_id,account_type,customer_segment,kyc_status,risk_rating_base,total_inflow,total_outflow,total_txn_count,drain_percentage,final_risk_level
0,C1278252907,CHECKING,RETAIL,VERIFIED,LOW,18211152.60,2779723.05,3,15.263850,CRITICAL
1,C968562623,SAVINGS,RETAIL,VERIFIED,MEDIUM,402336.76,2219179.46,4,551.572633,CRITICAL
2,C350004429,BUSINESS,RETAIL,VERIFIED,MEDIUM,474049.61,1589547.41,4,335.312460,CRITICAL
3,C750034968,BUSINESS,RETAIL,VERIFIED,LOW,392621.51,1569290.97,5,399.695618,CRITICAL
4,C26402899,BUSINESS,RETAIL,VERIFIED,LOW,198358.97,1305384.81,2,658.092150,HIGH
5,C1932870004,CHECKING,RETAIL,VERIFIED,MEDIUM,4525670.65,1265545.40,13,27.963710,CRITICAL
6,C1113713917,SAVINGS,RETAIL,VERIFIED,LOW,426202.69,1231455.53,3,288.936593,CRITICAL
8,C1592897762,CHECKING,RETAIL,VERIFIED,MEDIUM,1799462.64,1077152.65,8,59.859684,CRITICAL
9,C1829259083,SAVINGS,RETAIL,VERIFIED,MEDIUM,119690.91,967835.39,2,808.612275,HIGH
10,C1249550418,SAVINGS,RETAIL,VERIFIED,LOW,4695564.09,955541.52,17,20.349877,CRITICAL
